# 06. Urine Output Raw (소변량 원본 추출)

## 목적
코호트 환자의 소변량 데이터를 **1시간 단위**로 추출

## 입력
- `sepsis_cohort.csv`: 기본 코호트

## 출력
- `urine_raw.csv`: 시간별 소변량 (1 row = 1 patient × 1 hour)

## 전처리 범위
- [x] 데이터 추출 (outputevents)
- [x] 시간당 합산
- [x] 체중 데이터 병합 (mL/kg/hr 계산용)
- [ ] 슬라이딩 윈도우 적용 → merge 단계에서

## Item IDs
| Item ID | 설명 |
|---------|------|
| 226559 | Urine Out (Foley) - 소변줄 |
| 226560 | Urine Out (Void) - 자연 배뇨 |

In [1]:
from pathlib import Path
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
import duckdb
import pandas as pd
import os

DB_PATH = RAW_DIR / "mimic_total.duckdb"
INPUT_DIR = PROCESSED_DIR
OUTPUT_DIR = PROCESSED_DIR

con = duckdb.connect(DB_PATH)
print("=== 06. Urine Output Raw 추출 시작 ===")

=== 06. Urine Output Raw 추출 시작 ===


## Step 1: 코호트 로드

In [2]:
print("Step 1: 코호트 로드")

cohort_path = INPUT_DIR / "sepsis_cohort.csv"
df_cohort = pd.read_csv(cohort_path, parse_dates=['intime', 'outtime'])

print(f"✓ 코호트 로드 완료: {len(df_cohort):,}명")

con.execute("CREATE OR REPLACE TEMP TABLE cohort AS SELECT * FROM read_csv_auto(?)", [str(cohort_path)])


Step 1: 코호트 로드
✓ 코호트 로드 완료: 18,001명


## Step 2: 체중 데이터 추출

In [3]:
print("\nStep 2: 체중 데이터 추출")

weight_query = """
SELECT 
    c.stay_id,
    AVG(TRY_CAST(ce.valuenum AS DOUBLE)) as weight_kg
FROM cohort c
INNER JOIN chartevents ce ON c.stay_id = ce.stay_id
WHERE ce.itemid = '226512'  -- Admission Weight
  AND TRY_CAST(ce.valuenum AS DOUBLE) > 0
  AND TRY_CAST(ce.valuenum AS DOUBLE) < 500  -- 이상치 제외
GROUP BY c.stay_id
"""

df_weight = con.execute(weight_query).df()
print(f"✓ 체중 데이터: {len(df_weight):,}명")
print(f"  - 평균 체중: {df_weight['weight_kg'].mean():.1f} kg")


Step 2: 체중 데이터 추출
✓ 체중 데이터: 17,589명
  - 평균 체중: 82.8 kg


## Step 3: Urine Output 추출

In [4]:
print("\nStep 3: Urine Output 추출 (1시간 단위)")

urine_query = """
SELECT
    c.stay_id,
    date_trunc('hour', CAST(oe.charttime AS TIMESTAMP)) as charttime_h,
    SUM(TRY_CAST(oe.value AS DOUBLE)) as urine_ml
FROM cohort c
INNER JOIN outputevents oe ON c.stay_id = oe.stay_id
WHERE oe.itemid IN ('226559', '226560')  -- Foley, Void
  AND TRY_CAST(oe.value AS DOUBLE) > 0
  AND TRY_CAST(oe.value AS DOUBLE) < 5000  -- 이상치 제외 (시간당 5L 초과)
  -- ICU 체류 기간 내
  AND CAST(oe.charttime AS TIMESTAMP) >= c.intime
  AND CAST(oe.charttime AS TIMESTAMP) <= c.outtime
GROUP BY c.stay_id, charttime_h
ORDER BY c.stay_id, charttime_h
"""

print("  쿼리 실행 중...")
df_urine = con.execute(urine_query).df()

print(f"✓ Urine Output 추출 완료")
print(f"  - 총 행 수: {len(df_urine):,}개")
print(f"  - 고유 환자: {df_urine['stay_id'].nunique():,}명")


Step 3: Urine Output 추출 (1시간 단위)
  쿼리 실행 중...
✓ Urine Output 추출 완료
  - 총 행 수: 1,187,059개
  - 고유 환자: 17,535명


## Step 4: 체중 병합 및 mL/kg/hr 계산

In [5]:
print("\nStep 4: 체중 병합 및 mL/kg/hr 계산")

# 체중 병합
df_urine = df_urine.merge(df_weight, on='stay_id', how='left')

# 체중 없는 경우 전체 평균으로 대체
median_weight = df_weight['weight_kg'].median()
df_urine['weight_kg'] = df_urine['weight_kg'].fillna(median_weight)

# mL/kg/hr 계산
df_urine['urine_ml_kg_hr'] = df_urine['urine_ml'] / df_urine['weight_kg']

# 올리구리아 플래그 (0.5 mL/kg/hr 미만)
df_urine['oliguria_flag'] = (df_urine['urine_ml_kg_hr'] < 0.5).astype(int)

print(f"✓ 계산 완료")
print(f"  - 평균 urine: {df_urine['urine_ml'].mean():.1f} mL/hr")
print(f"  - 평균 urine/kg: {df_urine['urine_ml_kg_hr'].mean():.2f} mL/kg/hr")
print(f"  - 올리구리아 비율: {df_urine['oliguria_flag'].mean()*100:.1f}%")


Step 4: 체중 병합 및 mL/kg/hr 계산
✓ 계산 완료
  - 평균 urine: 133.1 mL/hr
  - 평균 urine/kg: 1.75 mL/kg/hr
  - 올리구리아 비율: 20.9%


## Step 5: 저장

In [6]:
print("\n" + "="*60)
print("CSV 저장")
print("="*60)

# 컬럼 정리
df_urine = df_urine[['stay_id', 'charttime_h', 'urine_ml', 'weight_kg', 'urine_ml_kg_hr', 'oliguria_flag']]

output_path = OUTPUT_DIR / "urine_raw.csv"
df_urine.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: urine_raw.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df_urine):,}개")
print(f"  - 경로: {output_path}")


CSV 저장

✓ 저장 완료: urine_raw.csv
  - 파일 크기: 67.53 MB
  - 행 수: 1,187,059개
  - 경로: DATA/processed/urine_raw.csv


In [7]:
print("\n=== 샘플 데이터 ===")
df_urine.head(10)


=== 샘플 데이터 ===


,stay_id,charttime_h,urine_ml,weight_kg,urine_ml_kg_hr,oliguria_flag
0,30000646,2194-04-29 05:00:00,700.0,79.9,8.760951,0
1,30000646,2194-04-29 08:00:00,1000.0,79.9,12.515645,0
2,30000646,2194-04-29 15:00:00,925.0,79.9,11.576971,0
3,30000646,2194-04-29 19:00:00,1000.0,79.9,12.515645,0
4,30000646,2194-04-29 23:00:00,750.0,79.9,9.386733,0
5,30000646,2194-04-30 07:00:00,800.0,79.9,10.012516,0
6,30000646,2194-04-30 10:00:00,1000.0,79.9,12.515645,0
7,30000646,2194-04-30 13:00:00,980.0,79.9,12.265332,0
8,30000646,2194-04-30 16:00:00,1540.0,79.9,19.274093,0
9,30000646,2194-04-30 21:00:00,800.0,79.9,10.012516,0


In [8]:
con.close()
print("\n=== 06. Urine Output Raw 추출 완료 ===")


=== 06. Urine Output Raw 추출 완료 ===
